In [3]:
import gurobipy as gp
from gurobipy import GRB, quicksum
import math
import time

class PDPTW_Optimizer:
    def __init__(self, file_path, time_limit=300):
        self.file_path = file_path
        self.time_limit = time_limit
        self.model = None
        self.solution = None
        self.runtime = 0
        self.gap = 0
        self.obj_val = 0
        
        # Data containers
        self.N = []      
        self.P = []      
        self.D = []      
        self.K = []      
        self.Start = 0
        self.End = 0
        self.coords = {}
        self.q = {}      
        self.e = {}      
        self.l = {}      
        self.d = {}      
        self.dist = {}   
        self.siblings = {} 
        self.Q_cap = 0   

    def read_li_lim_data(self, max_requests=None):
        """
        Parses Li & Lim benchmark files.
        Args:
            max_requests (int): If set, only loads the first N requests (for Instance Size analysis).
        """
        # print(f"Loading data from {self.file_path}...")
        with open(self.file_path, 'r') as f:
            lines = [line.strip() for line in f if line.strip()]

        header = lines[0].split()
        max_vehicles_file = int(header[0])
        self.Q_cap = int(header[1])

        node_data = []
        start_idx = 0
        for i, line in enumerate(lines):
            if line[0].isdigit():
                start_idx = i
                break
        
        # Parse all raw lines first
        raw_nodes = []
        for line in lines[start_idx:]:
            parts = line.split()
            if len(parts) < 9: continue
            raw_nodes.append({
                'id': int(parts[0]), 'x': float(parts[1]), 'y': float(parts[2]),
                'q': float(parts[3]), 'e': float(parts[4]), 'l': float(parts[5]),
                'd': float(parts[6]), 'pid': int(parts[7]), 'did': int(parts[8])
            })

        # --- INSTANCE SIZING (Critical for "Time vs Size" Analysis) ---
        original_depot = raw_nodes[0]
        customers = raw_nodes[1:]
        
        # Filter if max_requests is set
        if max_requests:
            # We must ensure we keep pairs. 
            # Li & Lim structure is usually: Depot, Pickup 1, Pickup 2... Delivery 1, Delivery 2...
            # We filter by finding the first N pickups and their specific siblings.
            pickups = [n for n in customers if n['q'] > 0][:max_requests]
            pickup_ids = {p['id'] for p in pickups}
            
            # Find corresponding deliveries
            deliveries = [n for n in customers if n['pid'] in pickup_ids]
            
            # Reconstruct customer list
            customers = pickups + deliveries
            # We sort by ID to keep logical order, though not strictly necessary
            customers.sort(key=lambda x: x['id'])

        # Create End Depot
        self.End = raw_nodes[-1]['id'] + 1000 # Make sure ID is unique and high
        end_depot = original_depot.copy()
        end_depot['id'] = self.End
        
        all_nodes = [original_depot] + customers + [end_depot]
        
        # Sets
        self.N = [n['id'] for n in all_nodes]
        self.P = [n['id'] for n in customers if n['q'] > 0]
        self.D = [n['id'] for n in customers if n['q'] < 0]
        self.Start = original_depot['id']
        
        # Dynamic Vehicle Sizing: 
        # For small subsets, we don't need 25 vehicles. 
        # Heuristic: 1 vehicle per 4 requests + buffer, or use file max.
        needed_vehicles = max(1, min(max_vehicles_file, math.ceil(len(self.P) / 2)))
        self.K = list(range(1, needed_vehicles + 1))

        # Parameters
        for n in all_nodes:
            nid = n['id']
            self.coords[nid] = (n['x'], n['y'])
            self.q[nid] = n['q']
            self.e[nid] = n['e']
            self.l[nid] = n['l']
            self.d[nid] = n['d']
            if n['did'] != 0: self.siblings[nid] = n['did']

        # Distances
        for i in self.N:
            for j in self.N:
                self.dist[i, j] = math.sqrt(
                    (self.coords[i][0] - self.coords[j][0])**2 + 
                    (self.coords[i][1] - self.coords[j][1])**2
                )

    def build_and_solve_model(self, verbose=True):
        m = gp.Model("GPDP_Savelsbergh")
        m.setParam('TimeLimit', self.time_limit)
        if not verbose:
            m.setParam('OutputFlag', 0)
        
        # Tight Big-M
        max_dist = max(self.dist.values())
        max_time = max(self.l.values())
        M_time = max_time + max_dist
        M_load = self.Q_cap + max(abs(v) for v in self.q.values())

        # Variables
        valid_arcs = [(i, j, k) for i in self.N for j in self.N for k in self.K if i != j]
        x = m.addVars(valid_arcs, vtype=GRB.BINARY, name="x")
        T = m.addVars(self.N, self.K, vtype=GRB.CONTINUOUS, lb=0, name="T")
        y = m.addVars(self.N, self.K, vtype=GRB.CONTINUOUS, lb=0, ub=self.Q_cap, name="y")

        # Objective
        m.setObjective(quicksum(self.dist[i,j] * x[i,j,k] for i,j,k in valid_arcs), GRB.MINIMIZE)

        # Constraints (Standard PDP Flow)
        m.addConstrs((quicksum(x[i,j,k] for k in self.K for j in self.N if i != j) == 1 for i in self.P), "VisitPickup")
        m.addConstrs((quicksum(x[j,i,k] for j in self.N if j != i) - quicksum(x[i,j,k] for j in self.N if j != i) == 0 
                      for k in self.K for i in self.P + self.D), "FlowCons")
        m.addConstrs((quicksum(x[self.Start,j,k] for j in self.P) <= 1 for k in self.K), "LeaveStart")
        m.addConstrs((quicksum(x[i,self.End,k] for i in self.D) == quicksum(x[self.Start,j,k] for j in self.P)
                      for k in self.K), "EnterEnd")
        
        m.addConstrs((T[i,k] + self.d[i] + self.dist[i,j] <= T[j,k] + M_time * (1 - x[i,j,k])
                      for i,j,k in valid_arcs), "TimeConsist")
        m.addConstrs((T[i,k] >= self.e[i] for k in self.K for i in self.N), "TW_Earliest")
        m.addConstrs((T[i,k] <= self.l[i] for k in self.K for i in self.N), "TW_Latest")
        
        m.addConstrs((y[i,k] + self.q[j] <= y[j,k] + M_load * (1 - x[i,j,k])
                      for i,j,k in valid_arcs), "LoadConsist")
        m.addConstrs((y[i,k] <= self.Q_cap for k in self.K for i in self.N), "CapMax")
        m.addConstrs((y[i,k] >= self.q[i] for k in self.K for i in self.N), "CapMin")
        m.addConstrs((y[self.Start,k] == 0 for k in self.K), "InitLoad")

        for i in self.P:
            sib = self.siblings[i]
            m.addConstrs((quicksum(x[i,j,k] for j in self.N if j != i) - 
                          quicksum(x[sib,j,k] for j in self.N if j != sib) == 0 
                          for k in self.K), f"Pairing_{i}")
            m.addConstrs((T[i,k] + self.d[i] + self.dist[i,sib] <= T[sib,k] for k in self.K), f"Prec_{i}")

        m.optimize()
        
        self.model = m
        self.solution = (x, T, y)
        self.runtime = m.Runtime
        self.obj_val = m.ObjVal if m.SolCount > 0 else float('inf')
        self.gap = m.MIPGap if m.SolCount > 0 else 1.0

    def print_diagnostics(self):
        """Prints metrics for the presentation analysis."""
        if self.model.SolCount > 0:
            print("\n" + "="*50)
            print("  OPTIMIZATION DIAGNOSTICS")
            print("="*50)
            print(f"  Instance Size (Requests) : {len(self.P)}")
            print(f"  Total Nodes              : {len(self.N)}")
            print(f"  Execution Time           : {self.runtime:.4f} seconds")
            print(f"  Optimality Gap           : {self.gap * 100:.2f}%")
            print(f"  Objective (Distance)     : {self.obj_val:.2f}")
            print("-" * 50)
            
            # Hardness metric: Average Time Window Width
            avg_tw = sum(self.l[i] - self.e[i] for i in self.P) / len(self.P)
            print(f"  Avg Time Window Width    : {avg_tw:.2f} (Narrower = Easier for exact)")
            print("="*50 + "\n")
        else:
            print(f"No solution found in {self.runtime:.2f}s")

In [ ]:
Hey everyone
I implemented cplex to solve the problem
Please take a look at the code, it was more comfortable for me that python sorryyy. You can ask chatGPT to explain, it is pretty simple to understand


/***************
 * OPL 22.1.1.0 Model
 * Author: oleksandra.yefimova
 * Creation Date: Dec 8, 2025 at 8:03:02 AM
 ***************/

// Sets 
range Nodes   = 0..6;  // 0 = depot, 1..6 = pickups & deliveries
range Clients = 1..6;  // all non-depot points

// Parameters
int Q = 2;             // vehicle capacity

// Demands: depot 0, pickups 1..3 (+1), deliveries 4..6 (-1)
int demand[Nodes] = [0, 1, 1, 1, -1, -1, -1];

// Distance matrix (symmetric, 0 on diagonal)
float dist[Nodes][Nodes] = [
  [0,   3.6, 6.4, 6.1, 3.2, 6.3, 6.4],  // from 0
  [3.6, 0,   3.2, 3.2, 2.2, 4.1, 2.8],  // from 1
  [6.4, 3.2, 0,   4.5, 3.6, 2.2, 1.4],  // from 2
  [6.1, 3.2, 4.5, 0,   5.4, 6.4, 3.2],  // from 3
  [3.2, 2.2, 3.6, 5.4, 0,   3.2, 4.1],  // from 4
  [6.3, 4.1, 2.2, 6.4, 3.2, 0,   3.6],  // from 5
  [6.4, 2.8, 1.4, 3.2, 4.1, 3.6, 0  ]   // from 6
];

// Big-M
int Mload = Q + 1;                 // To deactivate load rule when x=0 
int Mpos  = card(Clients);         // To deactivate ordering rule when x=0

// Decision variables 
// x[i][j] = 1 if we travel directly from node i to node j
dvar boolean x[Nodes][Nodes];	   

// L[i] = load of the vehicle after visiting node i
dvar float+ L[Nodes];

// u[i] = visit order position of client i in the route (1…6).
dvar int u[Clients];

// Objective: minimize total distance 
minimize
  sum (i in Nodes, j in Nodes : i != j) dist[i][j] * x[i][j];

// Constraints 
subject to {

  // Cannot drive from a node to itself
  forall (i in Nodes)
    x[i][i] == 0;

  // Each client has exactly one incoming and one outgoing arc
  forall (k in Clients) {
    sum (i in Nodes : i != k) x[i][k] == 1;  // exactly one predecessor
    sum (j in Nodes : j != k) x[k][j] == 1;  // exactly one successor
  }

  // Depot: exactly one departure and one arrival
  sum (j in Nodes : j != 0) x[0][j] == 1;    // leave depot once
  sum (i in Nodes : i != 0) x[i][0] == 1;    // return to depot once

  // Load must never go below 0 or above capacity Q.
  L[0] == 0;
  forall (i in Nodes)
    0 <= L[i] <= Q;

  // If we travel from i to j (if x[i][j] = 1), then L[j] = L[i] + demand[j]. 
  // If we do not travel frim i ti j (x[i][j] = 0), constraint becomes irrelevant (Big-M).
  forall (i in Nodes, j in Nodes : i != j) {
    L[j] >= L[i] + demand[j] - Mload * (1 - x[i][j]);
    L[j] <= L[i] + demand[j] + Mload * (1 - x[i][j]);
  }

  // Visit order variables must be between 1 and number of clients.
  forall (i in Clients)
    1 <= u[i] <= card(Clients);

  // If we travel from i to j, j must be visited after i (u[j] >= u[i] + 1). 
  // If not, constraint turns off
  forall (i in Clients, j in Clients : i != j) {
    u[j] >= u[i] + 1 - Mpos * (1 - x[i][j]);
  }

  // Pickup-delivery precedence (Each pickup must happen before its corresponding delivery)
  // Requests: (1 -> 4), (2 -> 5), (3 -> 6)
  u[1] + 1 <= u[4];
  u[2] + 1 <= u[5];
  u[3] + 1 <= u[6];
}